# INV-M365-CH Direct Full Surface Certification Universe Lock v1

## Purpose
Freeze the authoritative direct-certification universe for the repo-local M365 runtime.

## Supports
- `plan:m365-direct-full-surface-certification:R1`
- `F0` universe lock

## Expected Result
Produce a deterministic baseline showing the instruction surface, persona alias surface, capability surface, canonical crosswalk coverage, legacy direct actions still outside the v2 crosswalk, and the current legacy-stub action inventory.

In [ ]:
from __future__ import annotations

import json
import pathlib
import re

import yaml

ROOT = pathlib.Path.cwd()
REGISTRY = ROOT / "registry"
OUTPUT = ROOT / "artifacts" / "diagnostics" / "m365_direct_full_surface_certification.json"
ACTIONS_PY = ROOT / "src" / "ops_adapter" / "actions.py"
ROUTER_PY = ROOT / "src" / "provisioning_api" / "routers" / "m365.py"


In [ ]:
agents = yaml.safe_load((REGISTRY / "agents.yaml").read_text())["agents"]
agent_allowed = set()
for agent in agents.values():
    agent_allowed.update(agent.get("allowed_actions", []) or [])

capability_rows = yaml.safe_load((REGISTRY / "capability_registry.yaml").read_text()).get("actions", [])
capability_actions = {row["action"] for row in capability_rows}

router_text = ROUTER_PY.read_text()
supported_block = re.search(r"_SUPPORTED_ACTIONS\s*=\s*\{(.*?)\n\}", router_text, re.S)
assert supported_block, "_SUPPORTED_ACTIONS block not found"
instruction_actions = set(re.findall(r'\"([a-z0-9_]+)\"', supported_block.group(1)))

instruction_to_canonical = {}
capability_to_canonical = {}
agent_to_canonical = {}
canonical_sources = {}

for path in sorted(REGISTRY.glob("*_expansion_v2.yaml")):
    payload = yaml.safe_load(path.read_text()) or {}
    for instruction_alias, row in payload.get("supported_actions", {}).items():
        canonical = row["canonical_action_key"]
        instruction_to_canonical[instruction_alias] = canonical
        if row.get("capability_registry_alias"):
            capability_to_canonical[row["capability_registry_alias"]] = canonical
        if row.get("agent_alias"):
            agent_to_canonical[row["agent_alias"]] = canonical
        canonical_sources[canonical] = path.name

recipe_catalog = yaml.safe_load((REGISTRY / "cross_workload_automation_recipes_v2.yaml").read_text())
for instruction_alias, row in recipe_catalog.get("catalog_actions", {}).items():
    canonical = row["canonical_action_key"]
    instruction_to_canonical[instruction_alias] = canonical
    if row.get("capability_registry_alias"):
        capability_to_canonical[row["capability_registry_alias"]] = canonical
    if row.get("agent_alias"):
        agent_to_canonical[row["agent_alias"]] = canonical
    canonical_sources[canonical] = "cross_workload_automation_recipes_v2.yaml"

legacy_direct_actions = sorted(instruction_actions - set(instruction_to_canonical))
persona_aliases_without_crosswalk = sorted(agent_allowed - set(agent_to_canonical))


In [ ]:
actions_text = ACTIONS_PY.read_text()
stub_agents = [
    "it-operations-manager",
    "website-operations-specialist",
    "project-coordination-agent",
    "client-relationship-agent",
    "compliance-monitoring-agent",
    "recruitment-assistance-agent",
    "financial-operations-agent",
    "knowledge-management-agent",
]
legacy_stub_actions = {}
for agent in stub_agents:
    block = re.search(
        rf'if agent == "{re.escape(agent)}":(.*?)(?:\n\s*# ----|\n\s*if agent in \(|\n\s*raise ValueError)',
        actions_text,
        re.S,
    )
    assert block, f"stub block not found for {agent}"
    legacy_stub_actions[agent] = re.findall(r'if action == "([^"]+)"', block.group(1))

result = {
    "plan_id": "plan:m365-direct-full-surface-certification",
    "phase": "F0",
    "status": "universe_lock_complete",
    "counts": {
        "agents_total": len(agents),
        "instruction_surface_total": len(instruction_actions),
        "persona_alias_surface_total": len(agent_allowed),
        "capability_surface_total": len(capability_actions),
        "canonical_crosswalk_total": len(canonical_sources),
        "instruction_actions_mapped_to_canonical": len(instruction_actions & set(instruction_to_canonical)),
        "persona_aliases_mapped_to_canonical": len(agent_allowed & set(agent_to_canonical)),
        "capability_actions_mapped_to_canonical": len(capability_actions & set(capability_to_canonical)),
    },
    "legacy_direct_actions_without_crosswalk": legacy_direct_actions,
    "persona_aliases_without_crosswalk_total": len(persona_aliases_without_crosswalk),
    "persona_aliases_without_crosswalk_sample": persona_aliases_without_crosswalk[:80],
    "legacy_stub_inventory": {
        "stub_agents_total": len(legacy_stub_actions),
        "stub_actions_total": sum(len(v) for v in legacy_stub_actions.values()),
        "by_agent": legacy_stub_actions,
    },
    "notes": [
        "The direct repo certification universe is not yet a single surface; it requires the v2 expansion and recipe catalog crosswalk.",
        "The direct instruction surface still includes nine legacy actions outside the current expansion-backed canonical universe.",
        "A material subset of persona-facing actions remains legacy stub behavior in src/ops_adapter/actions.py and cannot yet be honestly certified as real M365 execution.",
    ],
}

OUTPUT.parent.mkdir(parents=True, exist_ok=True)
OUTPUT.write_text(json.dumps(result, indent=2) + "\n")
print(json.dumps(result, indent=2))
